# SDG 3 Indicator Text Classification
Person A notebook: EDA, label analysis, text characteristics, co-occurrence analysis, preprocessing, and data export.

# SDG 3 Indicator Text Classification
Person A notebook: EDA, label analysis, text characteristics, co-occurrence analysis, preprocessing, and data export.

## 1. Imports
Only the approved libraries are used. A helper script provides reusable functions.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
#import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import personA_eda_preprocessing as eda

sns.set_theme(style='whitegrid')
BASE_DIR = Path('.').resolve()
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Data Loading
Load train and test data and infer key columns dynamically.

In [8]:
train_df = eda.load_csv(BASE_DIR / 'Devex_train.csv')
test_df = eda.load_csv(BASE_DIR / 'Devex_test_questions.csv')

text_col = eda.detect_text_column(train_df)
label_cols = eda.detect_label_columns(train_df, text_col)
label_format = eda.detect_label_format(train_df, label_cols)
id_col = eda.detect_id_column(train_df, text_col, label_cols)
id_value = id_col if id_col else 'not_detected'

if text_col not in test_df.columns:
    raise ValueError(f"Text column '{text_col}' not found in test data.")

print('Text column:', text_col)
print('Label columns:', label_cols)
print('Label format:', label_format)
print('ID column:', id_value)
train_df.head()

FileNotFoundError: Missing file: C:\Users\HP\SDG indicator\SDG-Indicator-Text-Classification\Devex_train.csv

## 3. Dataset Overview
Summarize structure, missing values, and document length statistics.

In [ ]:
overview = pd.DataFrame({
    'column': train_df.columns,
    'dtype': [train_df[c].dtype for c in train_df.columns],
    'missing': [int(train_df[c].isna().sum()) for c in train_df.columns],
})
overview

,column,dtype,missing
0,Unique ID,int64,0
1,Type,object,0
2,Text,object,0
3,Label 1,object,0
4,Label 2,object,1360
5,Label 3,object,2257
6,Label 4,object,2683
7,Label 5,object,2853
8,Label 6,object,2936
9,Label 7,object,2974


In [ ]:
text_stats = eda.get_text_stats(train_df, text_col)
tokens_list = train_df[text_col].fillna('').astype(str).apply(eda.basic_tokenize)
vocab = eda.build_vocabulary(tokens_list)

dataset_summary = pd.DataFrame([
    {'metric': 'train_samples', 'value': len(train_df)},
    {'metric': 'test_samples', 'value': len(test_df)},
    {'metric': 'total_features_train', 'value': len(train_df.columns)},
    {'metric': 'total_features_test', 'value': len(test_df.columns)},
    {'metric': 'text_column', 'value': text_col},
    {'metric': 'id_column', 'value': id_value},
    {'metric': 'label_columns', 'value': len(label_cols)},
    {'metric': 'label_format', 'value': label_format},
    {'metric': 'unique_labels', 'value': None},
    {'metric': 'avg_doc_length_tokens', 'value': round(text_stats['word_count'].mean(), 2)},
    {'metric': 'median_doc_length_tokens', 'value': int(text_stats['word_count'].median())},
    {'metric': 'min_doc_length_tokens', 'value': int(text_stats['word_count'].min())},
    {'metric': 'max_doc_length_tokens', 'value': int(text_stats['word_count'].max())},
    {'metric': 'avg_char_count', 'value': round(text_stats['char_count'].mean(), 2)},
    {'metric': 'vocabulary_size', 'value': len(vocab)},
    {'metric': 'missing_values_train', 'value': int(train_df.isna().sum().sum())},
    {'metric': 'missing_values_test', 'value': int(test_df.isna().sum().sum())},
])
dataset_summary.to_csv(OUTPUT_DIR / 'dataset_summary.csv', index=False)
dataset_summary

,metric,value
0,train_samples,2995
1,test_samples,998
2,total_features_train,15
3,total_features_test,3
4,text_column,Text
5,id_column,Unique ID
6,label_columns,12
7,label_format,multi-column
8,unique_labels,None
9,avg_doc_length_tokens,489.13


In [ ]:
# Build label matrix and label names from label columns
labels_list = eda.build_label_lists(train_df, label_cols, label_format)
mlb = eda.MultiLabelBinarizer()
label_matrix = mlb.fit_transform(labels_list)
label_names = list(mlb.classes_)

# Compute label frequencies
label_counts = pd.Series(label_matrix.sum(axis=0), index=label_names).sort_values(ascending=False)

# Save label frequencies
label_freq_df = pd.DataFrame({
    'label': label_counts.index,
    'count': label_counts.values,
    'percentage': (label_counts.values / len(train_df) * 100).round(2)
})
label_freq_df.to_csv(OUTPUT_DIR / 'label_frequencies.csv', index=False)

print(f'Label matrix shape: {label_matrix.shape}')
print(f'Number of unique labels: {len(label_names)}')
label_freq_df.head(10)

Label matrix shape: (2995, 27)
Number of unique labels: 27


,label,count,percentage
0,3.b.2 - Total net official development assista...,1040,34.72
1,3.8.1 - Coverage of essential health services ...,529,17.66
2,3.4.1 - Mortality rate attributed to cardiovas...,483,16.13
3,3.b.3 - Proportion of health facilities that h...,394,13.16
4,"3.3.1 - Number of new HIV infections per 1,000...",371,12.39
5,3.2.1 - Under-5 mortality rate,249,8.31
6,3.c.1 - Health worker density and distribution,232,7.75
7,3.1.1 - Maternal mortality ratio,218,7.28
8,3.9.2 - Mortality rate attributed to unsafe wa...,217,7.25
9,3.d.1 - International Health Regulations (IHR)...,217,7.25


## 4. Label Analysis
Compute label frequencies, identify imbalance, and save the distribution plot.

In [ ]:
eda.plot_histogram(
    text_stats['word_count'],
    'Document Length Distribution (Words)',
    'Word Count',
    OUTPUT_DIR / 'document_length_histogram.png'
)
eda.plot_histogram(
    text_stats['char_count'],
    'Character Length Distribution',
    'Character Count',
    OUTPUT_DIR / 'character_length_histogram.png'
)
text_stats.describe()

,word_count,char_count,sentence_count
count,2995.000000,2995.000000,2995.000000
mean,489.131553,3409.484474,20.542905
std,656.619385,4648.366715,28.957719
min,2.000000,13.000000,1.000000
25%,19.000000,130.500000,1.000000
50%,213.000000,1463.000000,8.000000
75%,634.000000,4323.500000,26.000000
max,3961.000000,27632.000000,292.000000


In [ ]:
eda.plot_label_distribution(label_counts, OUTPUT_DIR / 'label_distribution.png')
imbalance_ratio = round(label_counts.max() / max(label_counts.min(), 1), 2)
most_common = label_counts.head(5).index.tolist()
rarest = label_counts.tail(5).index.tolist()
most_common, rarest, imbalance_ratio

/content/personA_eda_preprocessing.py:237: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


(['3.b.2 - Total net official development assistance to medical research and basic health sector',
  '3.8.1 - Coverage of essential health services (defined as the average coverage of essential services based on tracer interventions that include reproductive, maternal, newborn and child health, infectious diseases, non-communicable diseases and service capacity and access, among the general and the most disadvantaged population)',
  '3.4.1 - Mortality rate attributed to cardiovascular disease, cancer, diabetes or chronic respiratory disease',
  '3.b.3 - Proportion of health facilities that have a core set of relevant essential medicines available and affordable on a sustainable basis',
  '3.3.1 - Number of new HIV infections per 1,000 uninfected population, by sex, age and key populations'],
 ['3.9.3 - Mortality rate attributed to unintentional poisoning',
  '3.4.2 - Suicide mortality rate',
  '3.9.1 - Mortality rate attributed to household and ambient air pollution',
  '3.3.4 - Hepati

## 5. Text Characteristics
Generate document length histograms and a word cloud.

In [ ]:
eda.plot_histogram(
    text_stats['word_count']
    , 'Document Length Distribution (Words)'
    , 'Word Count'
    , OUTPUT_DIR / 'document_length_histogram.png'
)
eda.plot_histogram(
    text_stats['char_count']
    , 'Character Length Distribution'
    , 'Character Count'
    , OUTPUT_DIR / 'character_length_histogram.png'
)
text_stats.describe()

,word_count,char_count,sentence_count
count,2995.000000,2995.000000,2995.000000
mean,489.131553,3409.484474,20.542905
std,656.619385,4648.366715,28.957719
min,2.000000,13.000000,1.000000
25%,19.000000,130.500000,1.000000
50%,213.000000,1463.000000,8.000000
75%,634.000000,4323.500000,26.000000
max,3961.000000,27632.000000,292.000000


In [ ]:
normalized_text = train_df[text_col].fillna('').astype(str).apply(eda.normalize_text)
eda.plot_wordcloud(normalized_text.tolist(), OUTPUT_DIR / 'wordcloud.png')

## 6. Label Co-occurrence Analysis
Build the co-occurrence matrix and save the heatmap.

In [ ]:
cooccurrence_matrix = np.dot(label_matrix.T, label_matrix)
cooccurrence_df = pd.DataFrame(cooccurrence_matrix, index=label_names, columns=label_names)
eda.plot_cooccurrence_heatmap(cooccurrence_df, label_names, OUTPUT_DIR / 'label_cooccurrence_heatmap.png')
cooccurrence_df.head()

/content/personA_eda_preprocessing.py:276: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


,3.1.1 - Maternal mortality ratio,3.1.2 - Proportion of births attended by skilled health personnel,3.2.1 - Under-5 mortality rate,3.2.2 - Neonatal mortality rate,"3.3.1 - Number of new HIV infections per 1,000 uninfected population, by sex, age and key populations","3.3.2 - Tuberculosis incidence per 100,000 population","3.3.3 - Malaria incidence per 1,000 population","3.3.4 - Hepatitis B incidence per 100,000 population",3.3.5 - Number of people requiring interventions against neglected tropical diseases,"3.4.1 - Mortality rate attributed to cardiovascular disease, cancer, diabetes or chronic respiratory disease",...,3.8.2 - Proportion of population with large household expenditures on health as a share of total household expenditure or income,3.9.1 - Mortality rate attributed to household and ambient air pollution,"3.9.2 - Mortality rate attributed to unsafe water, unsafe sanitation and lack of hygiene (exposure to unsafe Water, Sanitation and Hygiene for All (WASH) services)",3.9.3 - Mortality rate attributed to unintentional poisoning,3.a.1 - Age-standardized prevalence of current tobacco use among persons aged 15 years and older,3.b.1 - Proportion of the target population covered by all vaccines included in their national programme,3.b.2 - Total net official development assistance to medical research and basic health sector,3.b.3 - Proportion of health facilities that have a core set of relevant essential medicines available and affordable on a sustainable basis,3.c.1 - Health worker density and distribution,3.d.1 - International Health Regulations (IHR) capacity and health emergency preparedness
3.1.1 - Maternal mortality ratio,218,39,135,80,30,15,17,1,1,14,...,9,1,7,1,0,6,29,16,9,3
3.1.2 - Proportion of births attended by skilled health personnel,39,96,29,24,4,4,3,0,0,0,...,6,0,5,0,0,5,9,9,34,3
3.2.1 - Under-5 mortality rate,135,29,249,79,38,17,14,2,6,22,...,8,2,9,4,3,5,23,15,11,7
3.2.2 - Neonatal mortality rate,80,24,79,138,16,7,6,4,6,11,...,4,1,2,2,3,7,20,10,6,4
"3.3.1 - Number of new HIV infections per 1,000 uninfected population, by sex, age and key populations",30,4,38,16,371,85,60,28,22,12,...,21,1,5,0,7,12,65,42,24,8


## 7. Preprocessing Pipeline
Apply lowercasing, HTML cleanup, stopword removal, and lemmatization while preserving domain acronyms.

In [ ]:
eda.ensure_nltk()
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

train_df['clean_text'] = train_df[text_col].apply(
    lambda x: eda.preprocess_text(x, stop_words, lemmatizer)
)
test_df['clean_text'] = test_df[text_col].apply(
    lambda x: eda.preprocess_text(x, stop_words, lemmatizer)
)
train_df[[text_col, 'clean_text']].head()

,Text,clean_text
0,Centers of Biomedical Research Excellence (COB...,center biomedical research excellence cobre ph...
1,Research on Regenerative Medicine <h2><strong>...,research regenerative medicine introduction su...
2,Catholic Health Association of India (CHAI): <...,catholic health association india chai catholi...
3,Quality Improvement Initiatives for Diabetes,quality improvement initiative diabetes
4,Provision of Thalassemia Drugs and Disposables...,provision thalassemia drug disposable backgrou...


## 8. Data Export and Summary
Write cleaned datasets and the report-ready Person A summary.

In [ ]:
train_output = BASE_DIR / 'devex_train_clean.csv'
test_output = BASE_DIR / 'devex_test_clean.csv'
train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

token_counter = (
    train_df['clean_text'].str.split().explode().value_counts().head(10)
)
top_terms = ', '.join(token_counter.index.tolist())

pairs = []
for i in range(len(label_names)):
    for j in range(i + 1, len(label_names)):
        count = cooccurrence_matrix[i, j]
        if count > 0:
            pairs.append((label_names[i], label_names[j], int(count)))
pairs.sort(key=lambda x: x[2], reverse=True)
top_pairs = '; '.join([f'{a} & {b} ({c})' for a, b, c in pairs[:5]])

most_common_str = ', '.join(most_common)
rarest_str = ', '.join(rarest)
avg_doc = round(text_stats['word_count'].mean(), 2)
median_doc = int(text_stats['word_count'].median())
min_doc = int(text_stats['word_count'].min())
max_doc = int(text_stats['word_count'].max())
avg_char = round(text_stats['char_count'].mean(), 2)
avg_sentence = round(text_stats['sentence_count'].mean(), 2)
missing_train = int(train_df.isna().sum().sum())
missing_test = int(test_df.isna().sum().sum())

no_pairs_text = 'No co-occurring labels detected.'

summary_lines = [
    '# Person A Summary',
    '',
    '## Dataset Description Findings',
    f'Train samples: {len(train_df)}',
    f'Test samples: {len(test_df)}',
    f'Detected text column: {text_col}',
    f'Detected ID column: {id_value}',
    f'Label columns: {len(label_cols)} (format: {label_format})',
    f'Unique labels: {len(label_names)}',
    f'Average document length: {avg_doc} tokens (median: {median_doc}, min: {min_doc}, max: {max_doc})',
    f'Average character count: {avg_char}',
    f'Vocabulary size (lowercased tokens): {len(vocab)}',
    f'Missing values (train/test): {missing_train} / {missing_test}',
    '',
    '## Label Imbalance Findings',
    f'Most common labels: {most_common_str}',
    f'Rarest labels: {rarest_str}',
    f'Imbalance ratio (max/min): {imbalance_ratio}',
    '',
    '## Text Characteristic Findings',
    f'Average sentence count: {avg_sentence}',
    f'Top domain terms (cleaned): {top_terms}',
    '',
    '## Co-occurrence Findings',
    f'Strongest co-occurring label pairs: {top_pairs if top_pairs else no_pairs_text}',
    '',
    '## Preprocessing Decisions',
    'Lowercasing, HTML tag removal, punctuation and special character filtering, whitespace normalization via token re-join.',
    'Tokenization with regex word boundaries.',
    'Stopword removal using NLTK English list.',
    'Lemmatization using WordNetLemmatizer.',
    'Preserved domain acronyms (e.g., SDG, WHO, HIV, TB, USAID) in uppercase.',
    'Numerical-only tokens removed.',
]

(BASE_DIR / 'personA_summary.md').write_text('\n'.join(summary_lines), encoding='utf-8')

3816

## 9. Outputs Checklist
Confirm generated files and their locations.

In [ ]:
generated_files = [
    OUTPUT_DIR / 'dataset_summary.csv',
    OUTPUT_DIR / 'label_frequencies.csv',
    OUTPUT_DIR / 'label_distribution.png',
    OUTPUT_DIR / 'document_length_histogram.png',
    OUTPUT_DIR / 'character_length_histogram.png',
    OUTPUT_DIR / 'wordcloud.png',
    OUTPUT_DIR / 'label_cooccurrence_heatmap.png',
    BASE_DIR / 'devex_train_clean.csv',
    BASE_DIR / 'devex_test_clean.csv',
    BASE_DIR / 'personA_summary.md',
]
[str(p) for p in generated_files]

['/content/outputs/dataset_summary.csv',
 '/content/outputs/label_frequencies.csv',
 '/content/outputs/label_distribution.png',
 '/content/outputs/document_length_histogram.png',
 '/content/outputs/character_length_histogram.png',
 '/content/outputs/wordcloud.png',
 '/content/outputs/label_cooccurrence_heatmap.png',
 '/content/devex_train_clean.csv',
 '/content/devex_test_clean.csv',
 '/content/personA_summary.md']